In [1]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
conn = sqlite3.connect('../data/lafc_content.db')

In [3]:
pd.read_sql('''
SELECT name
FROM sqlite_master
WHERE type='table';
''', conn)

,name
0,channel_snapshots
1,videos
2,teams
3,matches
4,standings_official
5,lafc_match_context
6,classified_videos


In [4]:
with open('../sql/classified_videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql(query, conn)
display(df.head())

,video_id,title,description,published_at,duration,dur_min,view_count,like_count,comment_count,format_family,...,home_away,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match
0,J8KtvKKDsBI,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,49.766667,1129,88,4,show,...,A,3.0,0.0,24.0,15.0,7.0,20.0,15.0,5.0,3.25
1,6IGiJLX6zIA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,0.283333,9169,901,20,match,...,A,3.0,0.0,24.0,15.0,7.0,20.0,15.0,5.0,3.12
2,dk5FTY2zHEI,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,2.166667,10782,1373,100,match,...,A,3.0,0.0,24.0,15.0,7.0,20.0,15.0,5.0,2.20
3,WGLkpecuCyA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,21.000000,2106,167,9,show,...,A,3.0,0.0,24.0,15.0,7.0,20.0,15.0,5.0,0.94
4,rjdbdSE3TNo,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,0.416667,2759,457,31,match,...,A,3.0,0.0,24.0,15.0,7.0,20.0,15.0,5.0,0.86


In [5]:
df.shape

(3571, 25)

In [6]:
fig = px.histogram(
    df, x='view_count',
    title="Views (raw) per video"
    )
fig.show()

In [7]:
df['log10_views'] = np.log10(df['view_count'])

In [8]:
fig = px.histogram(
    df, x='log10_views',
    nbins=80,
    title='Views (log10) per video', 
    labels={'log10_views': 'Views'}
    )
fig.update_xaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])
fig.show()

In [9]:
df['format_family'].value_counts()

format_family
social           1193
match            1138
show              686
media             269
feature           125
signing            63
behind_scenes      57
watch_party        40
Name: count, dtype: int64

In [10]:
df.groupby('format_family')['view_count'].median().sort_values()

format_family
show              798.5
watch_party       879.0
media            1072.0
social           1984.0
feature          2231.0
signing          2741.0
match            3085.0
behind_scenes    5449.0
Name: view_count, dtype: float64

<h3>Picked colors for the format family categories:<h3>

In [11]:
FAMILY_COLORS = {
    'behind_scenes': '#2a78d6',   # blue
    'feature':       '#eb6834',   # orange
    'match':         '#1baf7a',   # aqua
    'media':         '#eda100',   # yellow
    'show':          '#e87ba4',   # magenta
    'signing':       '#008300',   # green
    'social':        '#4a3aa7',   # violet
    'watch_party':   '#e34948',   # red
}

FIXED = sorted(FAMILY_COLORS)

In [12]:
order = df.groupby('format_family')['view_count'].median().sort_values().index.tolist()
n = df['format_family'].value_counts()

fig = px.box(
    df, x='format_family', y='log10_views',
    color='format_family',
    color_discrete_map=FAMILY_COLORS,
    category_orders={'format_family': order},
    title='Views (log 10) by format family',
    labels={'format_family': 'Format Family', 'log10_views': "Views"}
    )

for tr in fig.data:
    tr.legendrank = FIXED.index(tr.name)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [13]:
df['engagement_rate'] = (df['like_count'] + df['comment_count']) / df['view_count']

display(df[['title', 'engagement_rate']].sort_values('engagement_rate', ascending=False))

,title,engagement_rate
1988,Black & Gold Weekly | Turn The Page,0.250627
537,Acción LAFC Con Armando Aguayo | Ep. 72,0.180058
4,A Night To Remember | LAG vs LAFC,0.176876
1991,Black & Gold Weekly | Back On Track,0.164502
1994,Black & Gold Weekly | Running The Gauntlet,0.149462
...,...,...
730,Javairô Dilrosun scores his first goal in Blac...,0.004898
2337,Carlos Vela Opens The Scoring With A GOLAZO Ag...,0.003154
3549,Sports Arena Demolition Time-Lapse,0.001926
2167,"Power To Empower | LAFC, FLEX & Habitat for Hu...",0.001882


In [14]:
fig = px.histogram(
    df, x='engagement_rate',
    nbins=80,
    title= 'Engagement rate per video'
    )
fig.show()

In [15]:
df.groupby('format_family')['engagement_rate'].median().sort_values()

format_family
match            0.033600
signing          0.035737
behind_scenes    0.036441
feature          0.037037
social           0.042122
media            0.044821
watch_party      0.046931
show             0.052096
Name: engagement_rate, dtype: float64

In [16]:
order = df.groupby('format_family')['engagement_rate'].median().sort_values().index.tolist()
n = df['format_family'].value_counts()

fig = px.box(
    df, x='format_family', y='engagement_rate',
    color='format_family',
    color_discrete_map=FAMILY_COLORS,
    category_orders={'format_family': order},
    title='Engagement Rate by Format Family',
    labels={'format_family': 'Format Family', 'engagement_rate': 'Engagement Rate'},
    )

for tr in fig.data:
    tr.legendrank = FIXED.index(tr.name)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [17]:
fig = px.scatter(
    df, x='dur_min', y='view_count',
    log_y=True,
    opacity=0.3,
    trendline='ols',
    hover_data='title',
    title='View Count (log) by Duration in Minutes',
    labels={'view_count': 'View count', 'dur_min': 'Duration in Minutes'}
    )
fig.show()

print(f'R is {df['dur_min'].corr(df['view_count'])}')


R is -0.06509081197951162


In [18]:
df['log10_dur']   = np.log10(df['dur_min'])

In [19]:
fig = px.scatter(
    df, x='dur_min', y='view_count',
    log_y=True,
    log_x=True,
    opacity=0.3,
    trendline='ols',
    trendline_options=dict(log_x=True, log_y=True),   # ← fit in log space
    hover_data='title',
    title='View Count (log) by Duration in Minutes (log)',
    labels={'view_count': 'View count', 'dur_min': 'Duration in Minutes (log)'}
    )

fig.update_xaxes(
    tickvals=[0.25, 0.5, 1, 2, 5, 10, 30, 60, 120],
    ticktext=['15s', '30s', '1m', '2m', '5m', '10m', '30m', '1h', '2h'],
    title='Duration')

fig.show()

print(f"R is {df['log10_dur'].corr(df['log10_views']):.3f}")

R is -0.365


In [20]:
fig = px.scatter(
    df, x='log10_dur', y='engagement_rate',
    opacity=0.3,
    hover_data='title',
    trendline='lowess',
    trendline_options=dict(frac=0.3),
    title='Engagement Rate by Duration in Minutes',
    labels={'engagement_rate': 'Engagement Rate', 'dur_min': "Duration in Minutes"}
)

fig.update_xaxes(
    tickvals=[np.log10(v) for v in [0.25, 1, 5, 30, 120]],
    ticktext=['15s', '1m', '5m', '30m', '2h'],
    title='Duration')
fig.update_yaxes(tickformat='.1%')

fig.show()

In [ ]:
#Binning and adding the bin info back to the df.

BIN_EDGES  = [0, 2, 4, 8, 15, 31, np.inf]
BIN_LABELS = ['0-1', '2-3', '4-7', '8-14', '15-30', '30+']

df['days_bin'] = pd.cut(
    df['days_since_match'],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    right=False #So the bins go up to the higher edge, but don't include it.        
)

display(df[['title', 'days_since_match', 'days_bin']])

,title,days_since_match,days_bin
0,Inside LAFC | Episode 211 - A Strong Start,3.25,2-3
1,Sonny's goal from pitchside 🤳,3.12,2-3
2,Son Heung-Min | EVERY ANGLE of his derby goal ...,2.20,2-3
3,LAFC Weekly | Episode 15 | 2026,0.94,0-1
4,A Night To Remember | LAG vs LAFC,0.86,0-1
...,...,...,...
3566,Somos LAFC,NaN,NaN
3567,WE ARE LAFC,NaN,NaN
3568,John Thorrington announcement on SportsCenter,NaN,NaN
3569,Building Together: LAFC Stadium Workshop,NaN,NaN
